# Model Architecture Visualization

Ten notebook służy do wizualizacji i analizy struktury modeli używanych w projekcie StatsBomb Scout.

Dostępne modele:
- **LSTM**: Podstawowy model LSTM z dwiema warstwami
- **Attention LSTM**: LSTM z mechanizmem uwagi (attention)
- **BiGRU**: Bidirectional GRU z temporal attention pooling
- **Transformer**: Model oparty na architekturze Transformer

In [9]:
import sys
sys.path.append('src')

from src.ml.models.lstm import LSTMSequenceModel
from src.ml.models.attention_lstm import AttentionLSTMModel
from src.ml.models.bigru import build_seq_value_model
from src.ml.models.transformer import TransformerSequenceModel

from tensorflow import keras
import tensorflow as tf

# Parametry wejściowe (dostosuj według potrzeb)
INPUT_SHAPE = (30, 20)  # (max_sequence_length, num_features)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


## 1. LSTM Model

Podstawowy model LSTM składający się z:
- Warstwy maskującej (Masking) - ignoruje padding
- Dwóch warstw LSTM (128 jednostek, 64 jednostki)
- Warstw Dropout dla regularyzacji
- Warstwy Dense (64 jednostki) z aktywacją ReLU
- Wyjściowej warstwy Dense (1 jednostka) - regresja

In [10]:
# LSTM Model
lstm_model = LSTMSequenceModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.2
)
lstm_model.build()

print("=" * 80)
print("LSTM MODEL SUMMARY")
print("=" * 80)
lstm_model.model.summary()

LSTM MODEL SUMMARY


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 30, 20)    │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_3 (Masking) │ (None, 30, 20)    │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_3 (Any)         │ (None, 30)        │          0 │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 30, 128)   │     76,288 │ masking_3[0][0],  │
│                     │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 30, 128)   │          0 │ lstm_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 64)        │     49,408 │ dropout_7[0][0],  │
│                     │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 64)        │          0 │ lstm_5[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dropout_8[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 125,761 (491.25 KB)

 Trainable params: 125,761 (491.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Wizualizacja graficzna LSTM
keras.utils.plot_model(
    lstm_model.model,
    to_file='models/lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/lstm_architecture.png")

Diagram zapisany jako: models/lstm_architecture.png


## 2. Attention LSTM Model

LSTM z mechanizmem uwagi:
- Warstwy maskującej (Masking)
- Bidirectional LSTM (2 × 128 = 256 jednostek)
- Drugiej warstwy LSTM (128 jednostek)
- **Custom AttentionLayer** - oblicza wagi uwagi dla każdego kroku czasowego
- Warstwy Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [12]:
# Attention LSTM Model
attention_lstm_model = AttentionLSTMModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.2
)
attention_lstm_model.build()

print("=" * 80)
print("ATTENTION LSTM MODEL SUMMARY")
print("=" * 80)
attention_lstm_model.model.summary()

ATTENTION LSTM MODEL SUMMARY


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 30, 20)    │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_4 (Masking) │ (None, 30, 20)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_4 (Any)         │ (None, 30)        │          0 │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 30, 256)   │    152,576 │ masking_4[0][0],  │
│ (Bidirectional)     │                   │            │ any_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 30, 256)   │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 30, 128)   │    197,120 │ dropout_9[0][0],  │
│                     │                   │            │ any_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 30, 128)   │          0 │ lstm_7[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ [(None, 128),     │        158 │ dropout_10[0][0], │
│ (AttentionLayer)    │ (None, 30)]       │            │ any_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 64)        │          0 │ dense_6[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         65 │ dropout_11[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 358,175 (1.37 MB)

 Trainable params: 358,175 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Wizualizacja graficzna Attention LSTM
keras.utils.plot_model(
    attention_lstm_model.model,
    to_file='models/attention_lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/attention_lstm_architecture.png")

Diagram zapisany jako: models/attention_lstm_architecture.png


## 3. BiGRU Model

Bidirectional GRU z temporal attention:
- Warstwy maskującej (Masking)
- Bidirectional GRU (2 × 128 = 256 jednostek) z regularyzacją L2 i constraints
- Layer Normalization
- **TemporalAttentionPooling** - aggreguje sekwencję z wagami uwagi
- Dense (128 jednostek) z Batch Normalization
- Wyjście: wartość predykcji i wagi uwagi (opcjonalnie)

In [14]:
# BiGRU Model
bigru_model = build_seq_value_model(
    input_shape=INPUT_SHAPE,
    rnn_units=128,
    attn_hidden=64,
    dropout=0.2,
    recurrent_dropout=0.15,
    l2_reg=0.01,
    return_attention=True  # Zwróć również wagi uwagi
)

print("=" * 80)
print("BiGRU MODEL SUMMARY")
print("=" * 80)
bigru_model.summary()

BiGRU MODEL SUMMARY


Model: "bigru_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 30, 20)    │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_5 (Masking) │ (None, 30, 20)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_5 (Any)         │ (None, 30)        │          0 │ not_equal_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bigru               │ (None, 30, 256)   │    115,200 │ masking_5[0][0],  │
│ (Bidirectional)     │                   │            │ any_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 256)   │        512 │ bigru[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_pool           │ [(None, 256),     │     16,513 │ layer_normalizat… │
│ (TemporalAttention… │ (None, 30)]       │            │ any_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128)       │     32,896 │ attn_pool[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 128)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 30)        │          0 │ attn_pool[0][1]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │        129 │ dropout_13[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 165,762 (647.51 KB)

 Trainable params: 165,506 (646.51 KB)

 Non-trainable params: 256 (1.00 KB)

In [15]:
# Wizualizacja graficzna BiGRU
keras.utils.plot_model(
    bigru_model,
    to_file='models/bigru_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/bigru_architecture.png")

Diagram zapisany jako: models/bigru_architecture.png


## 4. Transformer Model

Model oparty na architekturze Transformer:
- Warstwy maskującej (Masking)
- Projekcja do d_model wymiarów
- **Positional Encoding** - dodaje informację o pozycji w sekwencji
- **Transformer Encoder Blocks** (domyślnie 2):
  - Multi-Head Self-Attention (4 głowice)
  - Feed-Forward Network (512 jednostek)
  - Layer Normalization i residual connections
- Global Average Pooling - agregacja sekwencji
- Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [17]:
# Transformer Model
transformer_model = TransformerSequenceModel(
    input_shape=INPUT_SHAPE,
    num_heads=4,
    d_model=128,
    ff_dim=512,
    num_blocks=2,
    dropout=0.1
)
transformer_model.build()

print("=" * 80)
print("TRANSFORMER MODEL SUMMARY")
print("=" * 80)
transformer_model.model.summary()

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [ ]:
# Wizualizacja graficzna Transformer
keras.utils.plot_model(
    transformer_model.model,
    to_file='models/transformer_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/transformer_architecture.png")

## 5. Porównanie liczby parametrów

Zestawienie wszystkich modeli z liczbą parametrów trenowalnych.

In [ ]:
import pandas as pd

# Zbierz statystyki wszystkich modeli
models_comparison = [
    {
        'Model': 'LSTM',
        'Total Parameters': lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in lstm_model.model.trainable_weights]),
        'Architecture': 'LSTM → LSTM → Dense'
    },
    {
        'Model': 'Attention LSTM',
        'Total Parameters': attention_lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in attention_lstm_model.model.trainable_weights]),
        'Architecture': 'Bi-LSTM → LSTM → Attention → Dense'
    },
    {
        'Model': 'BiGRU',
        'Total Parameters': bigru_model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in bigru_model.trainable_weights]),
        'Architecture': 'Bi-GRU → Temporal Attention → Dense'
    },
    {
        'Model': 'Transformer',
        'Total Parameters': transformer_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in transformer_model.model.trainable_weights]),
        'Architecture': 'Positional Encoding → Transformer Blocks → Pooling → Dense'
    }
]

df_comparison = pd.DataFrame(models_comparison)
df_comparison['Total Parameters'] = df_comparison['Total Parameters'].apply(lambda x: f"{x:,}")
df_comparison['Trainable Parameters'] = df_comparison['Trainable Parameters'].apply(lambda x: f"{x:,}")

print("\n" + "=" * 100)
print("MODEL COMPARISON")
print("=" * 100)
print(df_comparison.to_string(index=False))
print("=" * 100)

## 6. Analiza poszczególnych warstw

Szczegółowa analiza konkretnych warstw w wybranym modelu.

In [ ]:
# Przykład: Analiza warstw Attention LSTM
print("\nDetailed Layer Analysis - Attention LSTM Model:")
print("=" * 80)

for i, layer in enumerate(attention_lstm_model.model.layers):
    print(f"\nLayer {i}: {layer.name}")
    print(f"  Type: {type(layer).__name__}")
    print(f"  Output Shape: {layer.output_shape}")
    if hasattr(layer, 'units'):
        print(f"  Units: {layer.units}")
    if hasattr(layer, 'activation'):
        print(f"  Activation: {layer.activation}")
    
    # Liczba parametrów w warstwie
    trainable_params = sum([tf.size(w).numpy() for w in layer.trainable_weights])
    non_trainable_params = sum([tf.size(w).numpy() for w in layer.non_trainable_weights])
    
    if trainable_params > 0 or non_trainable_params > 0:
        print(f"  Trainable params: {trainable_params:,}")
        print(f"  Non-trainable params: {non_trainable_params:,}")

## 7. Testowe przewidywanie

Sprawdzenie czy model działa poprawnie na przykładowych danych.

In [ ]:
import numpy as np

# Wygeneruj przykładowe dane
batch_size = 2
sample_input = np.random.randn(batch_size, INPUT_SHAPE[0], INPUT_SHAPE[1]).astype(np.float32)

# Dodaj trochę padding'u (zera na końcu)
sample_input[:, -5:, :] = 0.0

print("\nTest Prediction:")
print("=" * 80)
print(f"Input shape: {sample_input.shape}")

# LSTM (pojedyncze wyjście)
lstm_pred = lstm_model.model.predict(sample_input, verbose=0)
print(f"\nLSTM prediction shape: {lstm_pred.shape}")
print(f"LSTM prediction values: {lstm_pred.flatten()}")

# Attention LSTM (dwa wyjścia: value + attention_weights)
attention_lstm_pred = attention_lstm_model.model.predict(sample_input, verbose=0)
print(f"\nAttention LSTM prediction:")
print(f"  Value shape: {attention_lstm_pred['value'].shape}")
print(f"  Value: {attention_lstm_pred['value'].flatten()}")
print(f"  Attention weights shape: {attention_lstm_pred['attention_weights'].shape}")
print(f"  Attention weights sum (should be ~1.0): {attention_lstm_pred['attention_weights'].sum(axis=1)}")

## 8. Eksport do LaTeX/dokumentacji

Przygotowanie tabel do publikacji naukowej.

In [ ]:
# Eksport do LaTeX
latex_table = df_comparison.to_latex(index=False, caption="Porównanie architektur modeli", label="tab:model_comparison")
print("\nLaTeX Table:")
print(latex_table)

# Zapisz do pliku
with open('models/model_comparison_table.tex', 'w') as f:
    f.write(latex_table)
print("\nTabela LaTeX zapisana jako: models/model_comparison_table.tex")